In [243]:
import pandas as pd
import numpy as np
import plotly.express as px
import sqlite3

In [244]:
df = pd.read_csv('online_retail_II.csv')

In [245]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


## Data prepearing

In [246]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [247]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


### Data Cleaning

In [248]:
df[df['Customer ID'].isna()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1066997,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom
1066998,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom
1066999,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom
1067000,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom


In [249]:
print(f"Частка дублікатів до: {round(df.duplicated().sum()/len(df), 3)}")
print(f"Частка пропущених значень до: \n{round(df.isna().sum()/len(df), 3)}")

df = (
    df
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
)

print(f"\nЧастка дублікатів після: {round(df.duplicated().sum()/len(df), 3)}")
print(f"Частка пропущених значень після: \n{round(df.isna().sum()/len(df), 3)}")

Частка дублікатів до: 0.032
Частка пропущених значень до: 
Invoice        0.000
StockCode      0.000
Description    0.004
Quantity       0.000
InvoiceDate    0.000
Price          0.000
Customer ID    0.228
Country        0.000
dtype: float64

Частка дублікатів після: 0.0
Частка пропущених значень після: 
Invoice        0.0
StockCode      0.0
Description    0.0
Quantity       0.0
InvoiceDate    0.0
Price          0.0
Customer ID    0.0
Country        0.0
dtype: float64


### Data Optimization

In [250]:
def invoice_prepearing(x):
    if x[0] == 'C':
        return -int(x[1:])
    else:
        return int(x)

In [251]:
df['Invoice'] = df['Invoice'].apply(invoice_prepearing)

In [252]:
df = df.astype({
    'Invoice': 'int32',
    'Quantity': 'int32',
    'Customer ID': 'uint16',
    'InvoiceDate': 'datetime64[s]'
})
df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

In [253]:
conn = sqlite3.connect('retail.db')
df.to_sql('retail', conn, if_exists="replace", index=False)

797885

In [254]:
query = "SELECT * FROM retail LIMIT 5"

pd.read_sql(query, conn)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


## Financial performance

### EDA

In [255]:
query = """
WITH t AS (
    SELECT Invoice, StockCode, InvoiceDate, Country, CustomerID, Price*Quantity AS Revenue
    FROM retail
    )
    
SELECT Country, SUM(Revenue) AS revenue_total, 100.0*SUM(Revenue)/(SELECT SUM(Revenue) FROM t) AS revenue_share, COUNT(DISTINCT CustomerID) AS num_customers, SUM(Revenue)/COUNT(DISTINCT CustomerID) AS ARPU
FROM t
GROUP BY Country
ORDER BY revenue_share DESC
"""

countries_stats = pd.read_sql(query, conn)

In [256]:
countries_stats.head()

,Country,revenue_total,revenue_share,num_customers,ARPU
0,United Kingdom,1.348251e+07,82.765576,5410,2492.145118
1,EIRE,5.735098e+05,3.520627,5,114701.952000
2,Netherlands,5.483307e+05,3.366059,23,23840.465217
3,Germany,4.119592e+05,2.528910,107,3850.085617
4,France,3.200463e+05,1.964680,95,3368.908000


In [257]:
print(f"Медіана кількості клієнтів: {countries_stats['num_customers'].median()}")
print(f"Медіана ARPU: {countries_stats['ARPU'].median()}")

Медіана кількості клієнтів: 6.0
Медіана ARPU: 2219.84


In [258]:
fig = px.histogram(countries_stats[countries_stats['num_customers']>=10], x='Country', y='revenue_share', title="Revenue Share by Countries")

fig.show()

In [259]:
fig = px.histogram(countries_stats[countries_stats['num_customers']>=10], x='Country', y='num_customers', title="Num Customers by Countries")

fig.show()

In [260]:
fig = px.histogram(countries_stats[countries_stats['num_customers']>=10], x='Country', y='ARPU', title="ARPU by Countries")

fig.show()

- Over 80% of total revenue comes from the UK, primarily driven by a significantly larger customer base (5,410 users vs median of 6), rather than higher ARPU
- Netherlands shows extremely high ARPU (~$23,840), likely caused by a small number of large transactions, indicating potential data skew

### Pareto Share

In [261]:
cursor = conn.cursor()

In [262]:
cursor.execute("DROP VIEW IF EXISTS customers_revenue")
cursor.execute(
"""
CREATE VIEW customers_revenue AS
SELECT Country, CustomerID, SUM(Price*Quantity) AS customer_revenue
FROM retail
GROUP BY Country, CustomerID
"""
)
conn.commit()

In [263]:
query = """
SELECT * FROM customers_revenue
"""

customers_revenue = pd.read_sql(query, conn)

In [264]:
query = """
WITH cumulative AS (
    SELECT Country, SUM(customer_revenue) OVER (PARTITION BY Country ORDER BY customer_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_sum
    FROM customers_revenue
    ),

share AS (
    SELECT Country, 1.0 * cumulative_sum/MAX(cumulative_sum) OVER (PARTITION BY Country) AS rev_share
    FROM cumulative
    )   

SELECT Country, 1.0 * MIN(num)/MAX(num) AS pareto_share
FROM (
    SELECT Country, rev_share, ROW_NUMBER() OVER (PARTITION BY Country ORDER BY rev_share) AS num
    FROM share
    ) AS t
WHERE rev_share >= 0.8
GROUP BY Country
"""

pareto_share_by_countries = pd.read_sql(query, conn)

In [265]:
countries_stats = (
    countries_stats
    .merge(pareto_share_by_countries, on='Country')
)

In [266]:
fig = px.histogram(countries_stats[countries_stats['num_customers']>=10], x='Country', y='pareto_share', title="Pareto Share by Countries")

fig.show()

- In the Netherlands, only 4.3% of customers generate ≥80% of revenue, indicating strong dependence on a few large buyers and making this market less stable

- Germany and France show strong ARPU ($3,850 and $3,368) combined with a more balanced customer distribution, making them attractive and stable markets for expansion

## Users Behavior

### EDA

In [267]:
cursor.execute("DROP VIEW IF EXISTS customers_revenue")
cursor.execute("DROP VIEW IF EXISTS orders")
cursor.execute("""
CREATE VIEW orders AS 
SELECT Country, CustomerID, COUNT(DISTINCT Invoice) AS num_orders
FROM retail
GROUP BY Country, CustomerID
""")
conn.commit()

In [268]:
query = """
SELECT Country, COUNT(DISTINCT CustomerID) AS num_customers, AVG(num_orders) AS mean_orders
FROM orders 
GROUP BY Country
ORDER BY num_customers DESC
"""

orders_stats = pd.read_sql(query, conn)

In [269]:
query = """
WITH t AS (
SELECT Country, Invoice, SUM(Quantity*Price) AS order_revenue 
FROM retail
WHERE Invoice > 0
GROUP BY Country, Invoice
    ),
aov AS(
SELECT Country, AVG(order_revenue) AS aov
FROM t
GROUP BY Country
    ),
orders_stats AS (
SELECT Country, COUNT(DISTINCT CustomerID) AS num_customers, AVG(num_orders) AS mean_orders
FROM orders 
GROUP BY Country
ORDER BY num_customers DESC
)

SELECT orders_stats.Country, orders_stats.num_customers, orders_stats.mean_orders, aov.aov
FROM orders_stats
JOIN aov
    ON orders_stats.Country = aov.Country
"""

orders_stats = pd.read_sql(query, conn)

In [270]:
orders_stats.head(10)

,Country,num_customers,mean_orders,aov
0,United Kingdom,5410,7.487061,428.940408
1,Germany,107,10.233645,538.681510
2,France,95,7.768421,568.027622
3,Spain,41,4.585366,703.457727
4,Belgium,29,6.310345,438.844430
5,Portugal,24,5.083333,597.363226
6,Netherlands,23,10.869565,2419.380306
7,Switzerland,22,5.454545,1111.799333
8,Sweden,19,6.736842,879.959808
9,Italy,17,5.411765,493.971846


In [271]:
fig = px.histogram(orders_stats[orders_stats['num_customers']>=10], x='Country', y='mean_orders', title='Median Orders vs Country')

fig.show()

In [272]:
fig = px.histogram(orders_stats[orders_stats['num_customers']>=10], x='Country', y='aov', title='AOV vs Country')

fig.show()

### RFM

#### Recency 

In [273]:
df['InvoiceDate'] = df['InvoiceDate'].dt.floor('D')

In [274]:
today = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [275]:
query = """
WITH t AS (
    SELECT Country, CustomerID, STRFTIME('%Y-%m-%d', InvoiceDate) AS InvoiceDate
    FROM retail
)
SELECT Country, CustomerID, JULIANDAY(DATE((SELECT MAX(InvoiceDate) FROM t), '+1 days')) - JULIANDAY(MAX(InvoiceDate)) AS recency
FROM t
GROUP BY Country, CustomerID  
"""

users_stats = pd.read_sql(query, conn)

In [276]:
users_stats['recency'].describe(percentiles=[.2, .4, .6, .8, 1])

count    5955.000000
mean      203.688161
std       211.851648
min         1.000000
20%        19.000000
40%        59.000000
50%        97.000000
60%       191.400000
80%       414.000000
100%      739.000000
max       739.000000
Name: recency, dtype: float64

In [277]:
users_stats['R_score'] = pd.qcut(
    users_stats['recency'],
    5,
    labels=[5,4,3,2,1]
)

In [278]:
fig = px.bar(users_stats['R_score'].value_counts())

fig.show()

#### Frequency

In [279]:
query = """
SELECT Country, CustomerID, COUNT(DISTINCT Invoice) AS num_orders
FROM retail
GROUP BY Country, CustomerID
"""

orders = pd.read_sql(query, conn)

In [280]:
users_stats = (
    users_stats
   .merge(orders, on=['Country', 'CustomerID'])
)

In [281]:
users_stats['num_orders'].describe(percentiles=[.2, .4, .6, .8, 1])

count    5955.000000
mean        7.535852
std        15.954215
min         1.000000
20%         1.000000
40%         3.000000
50%         4.000000
60%         5.000000
80%        10.000000
100%      510.000000
max       510.000000
Name: num_orders, dtype: float64

In [282]:
def f_score(x):
    if x == 1:
        return 1
    elif x <= 3:
        return 2
    elif x <= 5:
        return 3
    elif x <= 10:
        return 4
    else:
        return 5

In [283]:
users_stats['F_score'] = users_stats['num_orders'].apply(f_score)

In [284]:
fig = px.bar(users_stats['F_score'].value_counts())

fig.show()

#### Monetary

In [285]:
query = """
SELECT Country, CustomerID, SUM(Quantity*Price) AS customer_revenue
FROM retail
WHERE Quantity*Price > 0
GROUP BY Country, CustomerID  
"""

customers_revenue = pd.read_sql(query, conn)

In [286]:
users_stats = (
    users_stats
    .merge(customers_revenue, on=['Country', 'CustomerID'])
)

In [287]:
users_stats['customer_revenue'].describe(percentiles=[.2, .4, .6, .8, 1])

count      5891.000000
mean       2949.381135
std       14425.018154
min           2.950000
20%         285.560000
40%         608.790000
50%         862.420000
60%        1217.990000
80%        2896.690000
100%     580987.040000
max      580987.040000
Name: customer_revenue, dtype: float64

In [288]:
users_stats['M_score'] = pd.qcut(
    users_stats['customer_revenue'],
    5,
    labels=[1,2,3,4,5]
)

In [289]:
fig = px.bar(users_stats['M_score'].value_counts())

fig.show()

In [290]:
def classification(x):
    if x['R_score'] >= 4 and x['F_score'] >= 4 and x['M_score'] >= 4:
        return 'Champion'
    
    elif x['R_score'] >= 3 and x['F_score'] >= 4:
        return 'Loyal'
    
    elif x['R_score'] >= 3 and x['M_score'] >= 4 and x['F_score'] <= 3:
        return 'Big Spender'
    
    elif x['R_score'] >= 4 and x['F_score'] <= 2:
        return 'New'
    
    elif x['R_score'] <= 2 and x['F_score'] >= 3:
        return 'At Risk'
    
    else:
        return 'Lost'

#### Segmentation

In [291]:
users_stats['Class'] = users_stats.apply(classification, axis=1)

In [292]:
users_stats['Class'].value_counts(normalize=True)

Class
Lost           0.430487
Champion       0.213716
New            0.103208
At Risk        0.099813
Loyal          0.094891
Big Spender    0.057885
Name: proportion, dtype: float64

In [293]:
users_stats.to_excel('users_stats.xlsx')

In [294]:
rev_by_class = (
    users_stats
    .groupby(by='Class', as_index=False)
    .agg(
        class_revenue=('customer_revenue', lambda x: 100*(x.sum()/users_stats['customer_revenue'].sum()))
    )
)

In [295]:
users_stats

,Country,CustomerID,recency,R_score,num_orders,F_score,customer_revenue,M_score,Class
0,Australia,12386,338.0,2,2,2,401.90,2,Lost
1,Australia,12387,416.0,1,1,1,143.94,1,Lost
2,Australia,12388,16.0,5,8,4,3901.11,5,Champion
3,Australia,12389,403.0,2,3,2,1433.33,4,Lost
4,Australia,12392,592.0,1,1,1,234.75,1,Lost
...,...,...,...,...,...,...,...,...,...
5886,Unspecified,12470,690.0,1,1,1,211.95,1,Lost
5887,Unspecified,12743,135.0,3,2,2,540.13,2,Lost
5888,Unspecified,14265,110.0,3,4,3,1373.35,4,Big Spender
5889,Unspecified,16320,173.0,3,8,4,5628.99,5,Loyal


In [296]:
users_stats.to_excel('users_stats.xlsx')

In [297]:
fig = px.bar(users_stats['Class'].value_counts(), title='General RFM')

fig.show()

In [298]:
fig = px.histogram(rev_by_class, x='Class', y='class_revenue', title='Revenue by Classes')

fig.show()

In [299]:
fig = px.bar(users_stats[users_stats['Country'] == "United Kingdom"]['Class'].value_counts(), title='UK RFM')

fig.show()

In [300]:
fig = px.bar(users_stats[users_stats['Country'] == 'Germany']['Class'].value_counts(), title='Germany RFM')

fig.show()

In [301]:
fig = px.bar(users_stats[users_stats['Country'] == 'France']['Class'].value_counts(), title='France RFM')

fig.show()

In [302]:
fig = px.bar(users_stats[users_stats['Country'] == 'Netherlands']['Class'].value_counts(), title='Netherlands RFM')

fig.show()

In [303]:
(
    df
    .groupby(by=['Country', 'CustomerID'], as_index=False)
    .agg(
        lifetime=('InvoiceDate', lambda x: (x.max() - x.min()).days)
    )
)['lifetime'].median()

np.float64(224.0)

In [304]:
(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days

738

- A large share of customers falls into the "Lost" segment, likely due to shorter customer lifecycle (median 224 days) compared to the dataset period (738 days)
- Germany and France have a lower proportion of lost customers compared to the UK, indicating better retention and growth potential

## 

## Seasoning

### EDA

In [305]:
query = """
WITH t AS (
SELECT STRFTIME('%Y-%m', InvoiceDate) AS year_month, Quantity*Price AS Revenue, CustomerID
FROM retail
)
SELECT year_month, SUM(Revenue) AS total_revenue, COUNT(DISTINCT CustomerID) AS num_customers
FROM t
WHERE year_month < (SELECT MAX(year_month) FROM t)
GROUP BY year_month
"""

time_stats = pd.read_sql(query, conn)

In [306]:
exclude_months = ['2010-10', '2010-11', '2010-12', '2011-10', '2011-11']
def_mean_revenue = time_stats[~time_stats['year_month'].isin(exclude_months)]['total_revenue'].mean()
peak_mean_revenue = time_stats[time_stats['year_month'].isin(exclude_months)]['total_revenue'].mean()

In [307]:
print(f"Середнє значення доходів за Q1-Q3: {round(def_mean_revenue)}")
print(f"Середнє значення доходів за Q4: {round(peak_mean_revenue)}")
print(f"Приріст у Q4: {round((peak_mean_revenue/def_mean_revenue - 1)*100)}%")

Середнє значення доходів за Q1-Q3: 589758
Середнє значення доходів за Q4: 948608
Приріст у Q4: 61%


In [308]:
time_stats['year_month'] = time_stats['year_month'].astype(str) 

In [309]:
fig = px.line(time_stats, x='year_month', y='total_revenue', title='Revenue by Month')

fig.show()

In [310]:
fig = px.line(time_stats, x='year_month', y='num_customers', title='Num Customers by Month')

fig.show()

In [311]:
time_stats['MoM'] = (
    time_stats['total_revenue']
    .pct_change() * 100    
)

In [312]:
fig = px.line(time_stats, x='year_month', y='MoM', title='MoM')

fig.show()

In [313]:
query = """
WITH t AS (
SELECT Country, STRFTIME('%Y-%m', InvoiceDate) AS year_month, Quantity*Price AS Revenue, CustomerID
FROM retail
)

SELECT Country, year_month, COUNT(DISTINCT CustomerID) AS num_customers
FROM t
WHERE year_month < (SELECT MAX(year_month) FROM t)
GROUP BY Country, year_month
ORDER BY year_month
"""

time_customers = pd.read_sql(query, conn)

In [314]:
time_customers['year_month'] = time_customers['year_month'].astype(str)

In [315]:
fig = px.line(time_customers[time_customers['Country'] == 'United Kingdom'], x='year_month', y='num_customers', title="United Kingdom Customers Activity")

fig.show()

In [316]:
fig = px.line(time_customers[time_customers['Country'] == 'Germany'], x='year_month', y='num_customers', title="Germany Customers Activity")

fig.show()

In [317]:
fig = px.line(time_customers[time_customers['Country'] == 'France'], x='year_month', y='num_customers', title="France Customers Activity")

fig.show()

- Revenue shows strong seasonality, with Q4 revenue ~61% higher than other quarters
- Growth is driven mainly by an increase in active customers, indicating seasonal demand spikes
- Germany and France show steady growth in customer activity, while the UK appears to be a saturated market